# 🧪 PT-W1-D5 概念实验：Ontology 六维度 Gap 验证

> 配套阅读：同名 .md
> 实验目标：逐维度检查已有资产，量化缺口

## 第 1 格：六维度成熟度评分

In [ ]:
ontology_dimensions = {
    "Entity":    {"maturity": 3, "have": "terms 列表 + Object Ownership", "gap": "缺业务定义"},
    "Identity":  {"maturity": 2, "have": "表 ID + 编码规则", "gap": "缺业务身份层级"},
    "Relationship": {"maturity": 4, "have": "ADR-006 四类关系", "gap": "缺语义动词命名"},
    "State/Event":  {"maturity": 3, "have": "effect-registry 5 类", "gap": "缺完整状态机"},
    "Rule":     {"maturity": 2, "have": "BCM 域文件（部分）", "gap": "规则未显式化"},
    "Capability": {"maturity": 3, "have": "capabilities + BCM", "gap": "缺 Skill 映射"},
    "Policy":   {"maturity": 1, "have": "审批流代码", "gap": "几乎空白"},
}

print("Ontology 六维度 Gap Analysis：")
print(f"{'维度':<14} {'成熟度':<10} {'已有资产':<28} {'关键缺口'}")
print("-" * 80)
for dim, info in ontology_dimensions.items():
    stars = "★" * info["maturity"] + "☆" * (5 - info["maturity"])
    print(f"{dim:<14} {stars:<10} {info['have']:<28} {info['gap']}")

## 第 2 格：业务身份层级 vs 技术身份

In [ ]:
from dataclasses import dataclass

@dataclass
class BusinessIdentity:
    path: list
    semantic_label: str

@dataclass
class TechIdentity:
    table: str
    pk: int
    code: str

space_business = BusinessIdentity(
    path=["万象城(项目)", "A栋(楼宇)", "L2(楼层)", "L2-015(铺位)"],
    semantic_label="A栋二楼靠东第三个铺位"
)
space_tech = TechIdentity("shop", 12345, "B1-F2-015")

print("技术身份（AI 难以理解）：")
print(f"  table={space_tech.table}, id={space_tech.pk}, code={space_tech.code}")
print()
print("业务身份（AI 可推理）：")
print(f"  路径: {' → '.join(space_business.path)}")
print(f"  语义: {space_business.semantic_label}")
print()
print("用户说'A栋二楼那个奶茶店旁边'，AI 通过业务身份层级可定位到 L2-015")

## 第 3 格：FK vs 语义动词 — 关系命名对比

In [ ]:
fk_to_semantic = [
    ("合同.shop_id FK", "Lease **occupies** Space"),
    ("合同.tenant_id FK", "Lease **is-signed-by** Merchant"),
    ("铺位.building_id FK", "Space **is-located-in** Building"),
    ("账单.contract_id FK", "Bill **is-generated-by** Contract"),
    ("工单.space_id FK", "WorkOrder **targets** Space"),
]

print("FK（外键）→ 语义动词：")
print(f"{'技术 FK':<22} {'语义动词':<35}")
print("-" * 60)
for fk, semantic in fk_to_semantic:
    print(f"{fk:<22} {semantic}")

print("\nFK 告诉 AI '两个表有连接'")
print("语义动词告诉 AI '它们是什么关系' → 可推理")

## 第 4 格：可视化六维度雷达图

In [ ]:
from matplotlib import font_manager, pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("字体:", font_name)
dims = list(ontology_dimensions.keys())
scores = [ontology_dimensions[d]["maturity"] for d in dims]
labels = ["Entity","Identity","Relationship","State/Event","Rule","Capability","Policy"]

angles = np.linspace(0, 2*np.pi, len(dims), endpoint=False).tolist()
scores_plot = scores + [scores[0]]
angles_plot = angles + [angles[0]]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.fill(angles_plot, scores_plot, color="#3498db", alpha=0.25)
ax.plot(angles_plot, scores_plot, color="#3498db", linewidth=2)
ax.set_xticks(angles)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 5)
ax.set_title("MI CRE Ontology 六维度成熟度", fontsize=14, pad=20)
plt.tight_layout()
plt.savefig("/tmp/w1d5_radar.png", dpi=120)
plt.show()
print("最强：Relationship (★★★★☆)  最弱：Policy (★☆☆☆☆)")